# Offline Policy Evaluation


### Introduction

This notebook demonstrates the use of offline policy evaluation for MABs.

### Objectives

#### Evaluation:

Evaluate the performance of a MAB using multiple offline policy estimators.

In [1]:
import numpy as np
import pandas as pd
from sklearn.preprocessing import MinMaxScaler

from pybandits.cmab import CmabBernoulliCC
from pybandits.offline_policy_evaluator import OfflinePolicyEvaluator

%load_ext autoreload
%autoreload 2

/home/runner/.cache/pypoetry/virtualenvs/pybandits-vYJB-miV-py3.11/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


## Generate data

We first generate a binarly labeled data set, with a two dimensional feature space, and is not lineraly seprabale.
We then split the data set to a training data setm and a test data set.

In [2]:
n_samples = 1000
n_actions = 2
n_batches = 3
n_rewards = 1
n_groups = 2
n_features = 3

In [3]:
unique_actions = [f"a{i}" for i in range(n_actions)]
action_ids = np.random.choice(unique_actions, n_samples * n_batches)
batches = [i for i in range(n_batches) for _ in range(n_samples)]
rewards = [np.random.randint(2, size=(n_samples * n_batches)) for _ in range(n_rewards)]
action_true_rewards = {(a, r): np.random.rand() for a in unique_actions for r in range(n_rewards)}
true_rewards = [
    np.array([action_true_rewards[(a, r)] for a in action_ids]).reshape(n_samples * n_batches) for r in range(n_rewards)
]
groups = np.random.randint(n_groups, size=n_samples * n_batches)
action_costs = {action: np.random.rand() for action in unique_actions}
costs = np.array([action_costs[a] for a in action_ids])
context = np.random.rand(n_samples * n_batches, n_features)
action_propensity_score = {action: np.random.rand() for action in unique_actions}
propensity_score = np.array([action_propensity_score[a] for a in action_ids])
df = pd.DataFrame(
    {
        "batch": batches,
        "action_id": action_ids,
        "cost": costs,
        "group": groups,
        **{f"reward_{r}": rewards[r] for r in range(n_rewards)},
        **{f"true_reward_{r}": true_rewards[r] for r in range(n_rewards)},
        **{f"context_{i}": context[:, i] for i in range(n_features)},
        "propensity_score": propensity_score,
    }
)
contextual_features = [col for col in df.columns if col.startswith("context")]

## Generate Model

Using the cold_start method of CmabBernoulliCC, we can create a model to be used for offline policy evaluation.

In [4]:
action_ids_cost = {action_id: df["cost"][df["action_id"] == action_id].iloc[0] for action_id in unique_actions}

mab = CmabBernoulliCC.cold_start(action_ids_cost=action_ids_cost, n_features=len(contextual_features))

## OPE

Given the model and the OPE data from the logging policy, we can either evaluate the model using the logging policy, or update it with the logging policy data prior to the evaluation.

In [5]:
evaluator = OfflinePolicyEvaluator(
    split_prop=0.5,
    n_trials=10,
    fast_fit=True,
    scaler=MinMaxScaler(),
    ope_estimators=None,
    verbose=True,
    propensity_score_model_type="batch_empirical",
    expected_reward_model_type="gbm",
    importance_weights_model_type="logreg",
    batch_feature="batch",
    action_feature="action_id",
    reward_feature="reward_0",
    true_reward_feature="true_reward_0",
    contextual_features=contextual_features,
    group_feature="group",
    cost_feature="cost",
    propensity_score_feature="propensity_score",
)

In [6]:
evaluator.evaluate(mab=mab, logged_data=df, visualize=True, n_mc_experiments=1000)

  0%|          | 0/2 [00:00<?, ?it/s]

100%|██████████| 2/2 [00:00<00:00, 214.33it/s]


2026-11-26 20:54:02.101 | INFO     | pybandits.offline_policy_evaluator:_estimate_propensity_score:902 - Data batch-empirical estimation of propensity score.


2026-11-26 20:54:02.106 | INFO     | pybandits.offline_policy_evaluator:_estimate_expected_reward:951 - Data prediction of expected reward based on gbm model.


2026-11-26 20:54:02.959 | INFO     | pybandits.offline_policy_evaluator:estimate_policy:1068 - Data prediction of expected policy based on Monte Carlo experiments using 4 cores.


/opt/hostedtoolcache/Python/3.11.18/x64/lib/python3.11/multiprocessing/popen_fork.py:66: RuntimeWarning: os.fork() was called. os.fork() is incompatible with multithreaded code, and JAX is multithreaded, so this will likely lead to a deadlock.
  self.pid = os.fork()


... (2,138 outputs left out)

2026-11-26 20:54:19.180 | INFO     | pybandits.offline_policy_evaluator:_evaluate:1186 - Running OPE estimator 'sg-ipw' for reward 'reward_0'.


2026-11-26 20:54:19.389 | INFO     | pybandits.offline_policy_evaluator:_evaluate:1186 - Running OPE estimator 'switch-dr' for reward 'reward_0'.


Loading BokehJS ...

,value,lower,upper,std,estimator,objective
0,0.500455,0.467602,0.534325,0.016975,b-ipw,reward_0
1,0.489037,0.488426,0.489641,0.000311,dm,reward_0
2,0.503651,0.471654,0.536051,0.016349,dr,reward_0
3,0.489037,0.488438,0.489642,0.000309,dros-opt,reward_0
4,0.503651,0.472086,0.537940,0.016727,dros-pess,reward_0
5,0.502834,0.467877,0.536317,0.017433,ipw,reward_0
6,0.503217,0.469129,0.538206,0.017517,rep,reward_0
7,0.503671,0.471192,0.535851,0.016468,sndr,reward_0
8,0.503521,0.470479,0.538595,0.017465,snips,reward_0
9,0.503651,0.472222,0.536106,0.016253,sg-dr,reward_0


In [7]:
evaluator.update_and_evaluate(mab=mab, logged_data=df, visualize=True, n_mc_experiments=1000)

  0%|          | 0/2 [00:00<?, ?it/s]

100%|██████████| 2/2 [00:00<00:00, 426.84it/s]


2026-11-26 20:54:19.692 | INFO     | pybandits.offline_policy_evaluator:_update_mab:1300 - Offline policy update for <class 'pybandits.cmab.CmabBernoulliCC'>.


SVI:   0%|          | 0/1000 [00:00<?, ?it/s]

SVI:   0%|          | 1/1000 [00:00<08:40,  1.92it/s]

SVI:   0%|          | 1/1000 [00:00<08:40,  1.92it/s, loss=1808.2214]

... (4,218 outputs left out)

2026-11-26 20:54:54.708 | INFO     | pybandits.offline_policy_evaluator:_evaluate:1186 - Running OPE estimator 'sg-ipw' for reward 'reward_0'.


2026-11-26 20:54:54.920 | INFO     | pybandits.offline_policy_evaluator:_evaluate:1186 - Running OPE estimator 'switch-dr' for reward 'reward_0'.


Loading BokehJS ...

,value,lower,upper,std,estimator,objective
0,0.504475,0.465988,0.541537,0.019053,b-ipw,reward_0
1,0.489123,0.488518,0.489743,0.000315,dm,reward_0
2,0.508348,0.474020,0.542489,0.017603,dr,reward_0
3,0.489123,0.488526,0.489725,0.000306,dros-opt,reward_0
4,0.508348,0.472655,0.543470,0.017700,dros-pess,reward_0
5,0.507462,0.469237,0.547616,0.019919,ipw,reward_0
6,0.507908,0.469928,0.546729,0.019580,rep,reward_0
7,0.508375,0.473233,0.543448,0.017833,sndr,reward_0
8,0.508171,0.470686,0.546556,0.019478,snips,reward_0
9,0.508348,0.474922,0.544480,0.017723,sg-dr,reward_0
